In [10]:
import pandas as pd

In [12]:
df=pd.read_csv("IMDB Dataset.csv")

In [13]:

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [16]:
df.shape

(50000, 2)

In [18]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [20]:
df.drop_duplicates(inplace=True)

In [22]:
df.shape

(49582, 2)

#text preprocessing

In [25]:

#1.converting to lowercase

df["review"]=df["review"].str.lower()

In [27]:
#2.Removing the URLs

import re

# sample_text="abc is a word,abc"
# new_text=re.sub("abc","xyz",sample_text)

In [29]:
def remove_urls(text):
    text=re.sub(r"http\S+","",text)    #(pattern,replacement,str).....eg:https\\:www.google.com
    return text

df["review"]=df["review"].apply(remove_urls)
    
    

In [31]:
#3.remove punctuation
def remove_punctuations(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text)    #Exclude-----A-Z,a-z,0-9,\s(empty space)......baaki sab punctuation hai
    return text

df["review"]=df["review"].apply(remove_punctuations)
    

In [32]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [33]:
#4.remove html tags
def remove_html_tags(text):
    text=re.sub(r"<.*?>","",text)    
    return text

df["review"]=df["review"].apply(remove_html_tags)

In [36]:
#5.Remove stopwords
import nltk
nltk.download("pukt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Error loading pukt: Package 'pukt' not found in index
[nltk_data] Downloading package punkt_tab to
[nltk_data]     ~/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     ~/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [38]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

#sample_text="I love coding in python"
#tokens=word_tokenize(sample_text)

In [39]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text=text.replace(word ,"")

    return text       
df["review"]=df["review"].apply(remove_stopwords)    

In [41]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [43]:
#6.Stemming
#played-play

from nltk.stem import PorterStemmer

def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]

    tokens=word_tokenize(text)

    for token in tokens:
        stemmed_tokens=ps.stem(token)
        stemmed_words.append(stemmed_tokens)

    return " ".join(stemmed_words)   

df["review"]=df["review"].apply(stemming)      

In [44]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [45]:
#7.Encoding
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()

df["sentiment"]=le.fit_transform(df["sentiment"])

In [46]:
y=df["sentiment"]

In [47]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

In [48]:
#8.Vectorization

from sklearn.feature_extraction.text import TfidfVectorizer

tf=TfidfVectorizer(max_features=5000)
X=tf.fit_transform(df["review"])

In [49]:
X

<49582x5000 sparse matrix of type '<class 'numpy.float64'>'
	with 4057169 stored elements in Compressed Sparse Row format>

In [65]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)


In [67]:
X_train.shape

(39665, 5000)

In [69]:
X_test.shape

(9917, 5000)

In [71]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [73]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [74]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [76]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

In [86]:
import torch.nn as nn
import torch.optim as optim

In [88]:
class RNN ( nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [90]:
input_size=X.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

Train RNN

In [92]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.36048009991645813
epoch = 2/10 and loss = 0.3299647867679596
epoch = 3/10 and loss = 0.20962613821029663
epoch = 4/10 and loss = 0.29537221789360046
epoch = 5/10 and loss = 0.2868545949459076
epoch = 6/10 and loss = 0.22402139008045197
epoch = 7/10 and loss = 0.2588144838809967
epoch = 8/10 and loss = 0.1698310673236847
epoch = 9/10 and loss = 0.15844683349132538
epoch = 10/10 and loss = 0.30856823921203613


In [95]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.70132096400121
